# Week 4, Day 5: Brute force and approximation

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w004d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
count = 0
for a in range(1, 10):
    for b in range(1, 10):
        if a + b == 10:
            count += 1
print(count)

In [ ]:
heads = 35
legs = 94
for chickens in range(heads + 1):
    rabbits = heads - chickens
    if 2 * chickens + 4 * rabbits == legs:
        print(chickens, "chickens and", rabbits, "rabbits")

In [ ]:
for n in range(100, 1000):
    a = n // 100
    b = n // 10 % 10
    c = n % 10
    if a ** 3 + b ** 3 + c ** 3 == n:
        print(n)

In [ ]:
for code in range(1000):
    a, b, c = code // 100, code // 10 % 10, code % 10
    if a + b + c == 15 and a < b < c and code % 2 == 0:
        print(f"{code:03}")

In [ ]:
tries = 0
found = 0
for a in range(1, 101):
    for b in range(a + 1, 101):
        tries += 1
        c2 = a * a + b * b
        c = int(c2 ** 0.5)
        if c * c == c2 and c <= 100:
            found += 1
print("Found", found, "triples with c up to 100")
print("Pairs tried:", tries, "instead of", 100 ** 3, "triples")

> **Key idea.** Before writing a brute-force search, estimate how many candidates it will try. Up to a few million is fine. Billions need a smarter idea.

In [ ]:
x = 2
lo, hi = 0.0, max(1.0, x)
steps = 0
while hi - lo > 1e-10:
    mid = (lo + hi) / 2
    if mid * mid < x:
        lo = mid
    else:
        hi = mid
    steps += 1
print(lo, "after", steps, "steps")

> **Careful.** Never stop an approximation with `while mid * mid != x`. Floats are almost never exactly equal, so that loop might run for ever. Always stop on "close enough".

In [ ]:
x = 2
g = x
steps = 0
while abs(g * g - x) > 1e-12:
    g = (g + x / g) / 2
    steps += 1
    print(steps, g)

*Interactive figure on the website: Each row is one step. Bisection gains a digit every few steps; Newton's method doubles the correct digits each step once it's close..*

In [ ]:
total = 0
sign = 1
for k in range(100000):
    total += sign / (2 * k + 1)
    sign = -sign
print(4 * total)

In [ ]:
import random
inside = 0
darts = 100000
for _ in range(darts):
    x = random.random()
    y = random.random()
    if x * x + y * y <= 1:
        inside += 1
print(4 * inside / darts)

### Exercise: Heads and legs

Ask for the number of heads and legs with the prompts `Heads: ` and `Legs: `. The animals are chickens (2 legs) and cows (4 legs). Print `Chickens: ` and `Cows: ` on two lines, or `No answer` if no whole numbers work.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cases():\n    \"several farms\"\n    expect_output(\"Heads: 35\\nLegs: 94\\nChickens: 23\\nCows: 12\", inputs=[\"35\", \"94\"], echo=True)\n    expect_output(\"Heads: 10\\nLegs: 40\\nChickens: 0\\nCows: 10\", inputs=[\"10\", \"40\"], echo=True)\n    expect_output(\"Heads: 10\\nLegs: 41\\nNo answer\", inputs=[\"10\", \"41\"], echo=True)\n    expect_output(\"Heads: 3\\nLegs: 20\\nNo answer\", inputs=[\"3\", \"20\"], echo=True)")

### Exercise: Making change

How many different ways can you make an amount using any number of 5, 10 and 20 coins? Ask for the amount with the prompt `Amount: ` and print `Ways: ` and the count. For 20 there are 4 ways: 20; 10+10; 10+5+5; 5+5+5+5.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cases():\n    \"several amounts\"\n    for amount, ways in [(20, 4), (5, 1), (7, 0), (50, 12), (100, 36)]:\n        expect_output(f\"Amount: {amount}\\nWays: {ways}\", inputs=[str(amount)], echo=True)")

### Exercise: A digit puzzle

Find every two-digit number (10 to 99) that is equal to **three times the sum of its digits**. Print each on its own line.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the numbers\"\n    want = [n for n in range(10, 100) if n == 3 * (n // 10 + n % 10)]\n    expect_output(\"\\n\".join(str(n) for n in want))")

### Exercise: Cube root by bisection

Ask for a positive number with the prompt `x: ` and find its cube root by bisection, stopping when the interval is narrower than `1e-9`. Print `Cube root: ` and the answer rounded to 6 decimal places. Don't use `**` with a fraction.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cases():\n    \"several numbers\"\n    for x, r in [(\"27\", \"3.0\"), (\"2\", \"1.259921\"), (\"1000\", \"10.0\"), (\"0.5\", \"0.793701\")]:\n        expect_output(f\"x: {x}\\nCube root: {r}\", inputs=[x], echo=True)\ndef test_no_power():\n    \"uses bisection\"\n    assert \"** (1\" not in source and \"**(1\" not in source and \"** 0.\" not in source and \"**0.\" not in source, \"Find it by bisection.\"")

### Exercise: Count Newton's steps

Ask for a number with the prompt `x: ` (above 0) and use the Babylonian method, starting from the guess `g = x`, until `abs(g * g - x) < 1e-9`. Print `Steps: ` and how many steps it took, then `Root: ` and the root rounded to 6 places.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def steps_for(x):\n    g, s = x, 0\n    while abs(g * g - x) >= 1e-9:\n        g = (g + x / g) / 2\n        s += 1\n    return s, round(g, 6)\ndef test_cases():\n    \"several numbers\"\n    for x in [2.0, 10.0, 1000000.0, 0.25]:\n        s, r = steps_for(x)\n        expect_output(f\"x: {x:g}\\nSteps: {s}\\nRoot: {r}\", inputs=[f\"{x:g}\"], echo=True)")

### Exercise: How many terms?

Using the Leibniz series, find how many terms you need before `4 * total` is within `0.001` of `3.141592653589793`. Print `Terms: ` and the number of terms, then `Estimate: ` and the estimate rounded to 4 places.

In [ ]:
PI = 3.141592653589793

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the count\"\n    total, sign, k = 0.0, 1, 0\n    while True:\n        total += sign / (2 * k + 1)\n        sign = -sign\n        k += 1\n        if abs(4 * total - 3.141592653589793) < 0.001:\n            break\n    expect_output(f\"Terms: {k}\\nEstimate: {round(4 * total, 4)}\")")

### Bug hunt: Never close enough

This approximates √5 to 6 decimal places and happens to stop. But `mid * mid != x` is a dangerous test: with other numbers it may never become false, and the tiny limit only just rescues it. Make the stopping rule safe, using a tolerance of `1e-9` on the width of the interval, and keep the printed answer the same.

In [ ]:
x = 5
lo, hi = 0.0, 5.0
mid = (lo + hi) / 2
while mid * mid != x and hi - lo > 1e-15:
    if mid * mid < x:
        lo = mid
    else:
        hi = mid
    mid = (lo + hi) / 2
print(round(mid, 6))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"prints the root\"\n    expect_output(\"2.236068\")\ndef test_tolerance():\n    \"stops on a tolerance, not on ==\"\n    assert \"!=\" not in source and \"1e-9\" in source, \"Loop while hi - lo > 1e-9.\"")

### Bug hunt: One short

This should print every number up to and **including** 105 that divides by 3, 5 and 7. The smallest such number is 3 × 5 × 7 = 105, but the program prints nothing. Fix the range.

In [ ]:
for n in range(1, 105):
    if n % 3 == 0 and n % 5 == 0 and n % 7 == 0:
        print(n)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"prints 105\"\n    expect_output(\"105\")")

### Puzzle: Newton's method

Arrange the lines into the Babylonian square root. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Trace bisection

Trace three steps of bisection for √10, starting with `lo = 0` and `hi = 10`. After each step, write down `lo` and `hi`.

```python
x = 10
lo, hi = 0, 10
# repeat 3 times:
mid = (lo + hi) / 2
if mid * mid < x:
    lo = mid
else:
    hi = mid
```

In [ ]:
after_1 = (___, ___)
after_2 = (___, ___)
after_3 = (___, ___)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_1():\n    \"step 1\"\n    assert after_1 == (0, 5), \"mid is 5, and 25 is not < 10, so hi becomes 5.\"\ndef test_2():\n    \"step 2\"\n    assert after_2 == (2.5, 5), \"mid is 2.5, and 6.25 is < 10, so lo becomes 2.5.\"\ndef test_3():\n    \"step 3\"\n    assert after_3 == (2.5, 3.75), \"mid is 3.75, and 14.0625 is not < 10, so hi becomes 3.75.\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: divisible by everything

What is the smallest positive number that divides exactly by every whole number from 1 to `n`? For `n = 10` it's 2520.

Ask for `n` with the prompt `n: ` and print `Smallest: ` and the answer. The checks go up to `n = 30`, and the answer then has 14 digits, so a brute-force search that tries every number would never finish. Find a faster way.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def answer(n):\n    r = 1\n    for k in range(2, n + 1):\n        a, b = r, k\n        while b:\n            a, b = b, a % b\n        r = r * k // a\n    return r\ndef test_small():\n    \"small n\"\n    for n in [1, 5, 10]:\n        expect_output(f\"n: {n}\\nSmallest: {answer(n)}\", inputs=[str(n)], echo=True)\ndef test_big():\n    \"n = 30\"\n    expect_output(f\"n: 30\\nSmallest: {answer(30)}\", inputs=[\"30\"], echo=True)")

## Warm-up

That's a tiny brute-force search: try all 81 pairs, keep the ones that work. Computers are so fast that trying everything is often the simplest correct solution.

## Brute force: try everything

A farmer has chickens and rabbits. Together they have 35 heads and 94 legs. How many of each?

You could set up simultaneous equations. Or you could let the computer try every possible number of chickens:

That's **brute force**, also called **exhaustive search**. It has three parts:

1. **The candidates**: every possible answer, in a range you can loop over.
2. **The test**: a condition that says whether a candidate works.
3. **What to do with a match**: print it, count it, or stop at the first one.

It's reliable: if the answer is among the candidates, brute force finds it. The only question is whether there are too many candidates.

Only four numbers in the whole range have this property. Brute force found them in an instant by trying 900 candidates.

### A lock with clues

The clues: the digits add up to 15, they go up from left to right, and the code is even. The loop tries all 1,000 codes and prints the two that fit.

## Making the search smaller

Brute force gets slow when the candidates multiply. Three nested loops over 1 to 1,000 try a **billion** combinations. So before you search, shrink the range:

- **Use what you know.** In the chickens puzzle, `rabbits = heads - chickens` means one loop instead of two.
- **Use limits.** If `a² + b² = c²` and `c ≤ 100`, then `a` can't be more than 100 either.
- **Order the candidates.** Insisting `a < b` halves the pairs and removes duplicates.
- **Stop early.** If you only need one answer, `break` at the first.

Working out `c` from `a` and `b` instead of looping over it turned a million tries into about five thousand.

## Approximation: closer and closer

What's √2? It's about 1.41421356..., and its digits never end, so no computer can hold it exactly. But you can get as close as you like. That's **approximation**.

### Bisection

√2 is between 1 and 2, because 1² = 1 is too small and 2² = 4 is too big. Try the **middle**, 1.5: 1.5² = 2.25, too big, so the answer is between 1 and 1.5. Try the middle of *that*. Every step halves the range where the answer can be:

The loop stops when the range is narrower than `1e-10` (that's 0.0000000001). That number is the **tolerance**: how close is close enough.

### Newton's method

About 4,000 years ago, Babylonian mathematicians used a faster trick. If `g` is a guess for √x, then `x / g` is off by the same factor in the other direction, so the **average** of `g` and `x / g` is a better guess. Repeat:

Five steps. This is a special case of **Newton's method**, one of the most important algorithms in science and engineering: it's how calculators compute square roots, and how machine learning models are trained, in a more general form. Compare the two methods:

## Estimating π

### With a series

The number π can be written as a never-ending sum: π = 4 × (1 − 1/3 + 1/5 − 1/7 + 1/9 − ...). Add up more and more terms, and you get closer:

A hundred thousand terms give only five correct digits, so this series is beautiful but slow. Mathematicians have found series that gain many digits per term.

### With random points

Throw darts at random into a 1 × 1 square. The quarter-circle of radius 1 covers π/4 of the square, so about π/4 of the darts land inside it:

`random.random()` gives a random float from 0 up to (not including) 1. Run it a few times: the answer wobbles. Methods that use randomness like this are called **Monte Carlo** methods, after the casino, and they're used everywhere from physics to finance.

## Practice

## Quiz

## Daily challenge

## Before you go

Think of a puzzle you could solve by brute force, and estimate how many candidates it would try. Then explain, in two sentences, why approximation loops stop on a tolerance and never on `==`. Tomorrow is the Forge: a bigger project that brings the week together, and the weekly quiz.